In [1]:
import pandas as pd
import numpy as np


# 1. LOAD THE GROUND TRUTH DATA

# Manually load the different generated data sets
print("Loading ground truth dataset...")
df_blackout = pd.read_csv('MICE_wide_1jamnew_10.csv')

# Convert Timestamp to a true datetime object so we can filter exactly
df_blackout['Timestamp'] = pd.to_datetime(df_blackout['Timestamp'])


# 2. DEFINE MANUAL BLACKOUT PARAMETERS

target_columns = [
    'Spd_S4_dir_a', 'Vol_S4_dir_a', 
    'Spd_S4_dir_b', 'Vol_S4_dir_b'
]

# The 4 curated "Perfect Portfolio" Blackouts for Pattern B
# Dates are specifically chosen in August 2026 to avoid the July 25th Pattern C jam
manual_blackouts = [
    {"name": "Late Night Flatline (Tue)", "start": "2026-06-09 01:00:00", "end": "2026-06-09 04:30:00"},
    {"name": "Commuter Peak (Wed)",       "start": "2026-06-17 06:30:00", "end": "2026-06-17 10:00:00"},
    {"name": "Complex Peak (Fri)",        "start": "2026-08-14 15:30:00", "end": "2026-08-14 19:30:00"},
    {"name": "Stochastic Midday (Sun)",   "start": "2026-08-23 10:30:00", "end": "2026-08-23 14:00:00"}
]

print(f"Injecting {len(manual_blackouts)} curated continuous blackouts on Sensor 4...\n")


# 3. INJECT THE MANUAL BLACKOUTS

for b in manual_blackouts:
    start_dt = pd.to_datetime(b["start"])
    end_dt = pd.to_datetime(b["end"])
    
    # Create a mask for rows that fall exactly inside this time window 
    # Using < end_dt so we don't accidentally blank an extra minute
    mask = (df_blackout['Timestamp'] >= start_dt) & (df_blackout['Timestamp'] < end_dt)
    
    # Inject the NaNs
    df_blackout.loc[mask, target_columns] = np.nan
    
    # Calculate how many minutes were actually blanked out
    missing_count = mask.sum()
    print(f"Injected {b['name']}: {b['start']} to {b['end']} ({missing_count} minutes)")


# 4. VALIDATION & EXPORT

total_missing_minutes = df_blackout[target_columns[0]].isna().sum()
print(f"\nTotal missing minutes injected on Sensor 4: {total_missing_minutes}")

# Export the new curated dataset
# change the csv file for each of the data generated.
df_blackout.to_csv('patternB_curated_blackouts.csv', index=False)
print("Saved to 'patternB_curated_blackouts.csv'")

Loading ground truth dataset...
Injecting 4 curated continuous blackouts on Sensor 4...

Injected Late Night Flatline (Tue): 2026-06-09 01:00:00 to 2026-06-09 04:30:00 (210 minutes)
Injected Commuter Peak (Wed): 2026-06-17 06:30:00 to 2026-06-17 10:00:00 (210 minutes)
Injected Complex Peak (Fri): 2026-08-14 15:30:00 to 2026-08-14 19:30:00 (240 minutes)
Injected Stochastic Midday (Sun): 2026-08-23 10:30:00 to 2026-08-23 14:00:00 (210 minutes)

Total missing minutes injected on Sensor 4: 870
Saved to 'patternB_curated_blackouts.csv'
